# 04 Spatial

## From 01–03

- NO₂ at 15 stations, PM2.5 at 12, hourly, 2025
- NO₂ annual means 7.6 to 26.3 µg/m³, PM2.5 8.2 to 13.6
- Temporal coverage 98.5–99.9 % after drops

## Question

Earlier notebooks show NO₂ varies by location. That is a statement about 15
measured points across 892 km². This notebook asks how far a measurement
travels: if a station did not exist, how closely could its value be predicted
from the others?

## Method

Leave-one-out inverse distance weighting. Hide each station in turn, estimate
its annual mean from the remaining stations weighted by 1/d², compare to the
measured value, record the error. Repeat for all stations, both pollutants.

In [5]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

# One colour system across notebooks.
# Hue = pollutant. Blue/orange is CVD safe.
COLOUR = {"no2": "#4c72b0", "pm2": "#dd8452"}
PALE   = {"no2": "#b8c6de", "pm2": "#f0cab5"}
CMAP   = {"no2": "mako_r", "pm2": "flare"}

CURRENT = {"no2": 40, "pm2": 25}    # annual mean, in force 2025
LIMIT   = {"no2": 20, "pm2": 10}    # annual mean, EU 2030
WHO     = {"no2": 10, "pm2": 5}

# $_2$ renders the subscript. The plain ₂ character draws as a box in figures.
LABEL = {
    "no2": "NO$_2$ · mainly traffic exhaust · airways",
    "pm2": "PM2.5 · fine particles · heart and blood vessels",
}

NAME = {
    "mc010": "Amrumer Str., Wedding",
    "mc027": "Schichauweg, Marienfelde",
    "mc032": "Grunewald, Charlottenburg-Wilmersdorf",
    "mc042": "Nansenstr., Neukölln",
    "mc077": "Buch, Pankow",
    "mc085": "Friedrichshagen, Treptow-Köpenick",
    "mc117": "Schildhornstr., Steglitz",
    "mc124": "Mariendorfer Damm, Mariendorf",
    "mc144": "Silbersteinstr., Neukölln",
    "mc145": "Frohnau, Reinickendorf",
    "mc171": "Brückenstr., Mitte",
    "mc174": "Frankfurter Allee, Friedrichshain",
    "mc190": "Leipziger Str., Mitte",
    "mc221": "Karl-Marx-Str., Neukölln",
    "mc282": "Karlshorst, Lichtenberg",
}

TYPE = {
    "mc010": "background", "mc042": "background",
    "mc171": "background", "mc282": "background",
    "mc027": "suburb", "mc032": "suburb", "mc077": "suburb",
    "mc085": "suburb", "mc145": "suburb",
    "mc117": "traffic", "mc124": "traffic", "mc144": "traffic",
    "mc174": "traffic", "mc190": "traffic", "mc221": "traffic",
}

# Cleaned table from 01. Drops and fireworks flag already applied.
# CSV stores the offset as text, so parse to UTC then convert, as in 01.
ROOT = Path.cwd().parent
df = pd.read_csv(ROOT / "data" / "processed" / "hourly.csv")
df["datetime"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("Europe/Berlin")
df["fireworks"] = df["fireworks"].astype(bool)      # survives CSV as text otherwise

print(df.shape)

(236180, 7)


In [6]:
# Station coordinates are not in the hourly data. One request to /api/stations,
# cached like the measurement files so the notebook reruns without the network.
import json
from urllib.request import urlopen

RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
path = RAW / "stations.json"

if path.exists():
    stations = json.loads(path.read_text())
else:
    with urlopen("https://luftdaten.berlin.de/api/stations?active=true") as r:
        text = r.read().decode()
    path.write_text(text)
    stations = json.loads(text)

print(type(stations), len(stations))
print(json.dumps(stations[0] if isinstance(stations, list) else stations, indent=2)[:800])

<class 'list'> 15
{
  "name": "010 Wedding",
  "code": "mc010",
  "codeEu": "DEBE010",
  "address": "13353 Berlin, Amrumer Str./Limburger Str.",
  "lat": "52.54291000",
  "lng": "13.34926000",
  "active": true,
  "stationgroups": [
    "background"
  ],
  "measuringStart": "1984-11-01T00:00:00+01:00",
  "measuringEnd": null,
  "measuringHeight": null,
  "url": "https://gdi.berlin.de/data/ua_luftguetemessnetz/pictures/MC010-Doku.pdf",
  "information": "Die Station befindet sich an der Beuth Hochschule f\u00fcr Technik. Hat MP910 ersetzt.",
  "components": [
    "pm10_1h",
    "pm10_24h",
    "pm10_24hg",
    "pm10_1m",
    "pm10_1y",
    "no_1h",
    "no_24h",
    "no_1m",
    "no_1y",
    "no2_1h",
    "no2_24h",
    "no2_1m",
    "no2_1y",
    "nox_1h",
    "nox_24h",
    "nox_1m",
    "nox_1y",
    "chb_1


In [7]:
# Build a station table from the API: coordinates, group, and whether a
# measuring height is published. Strings to floats; lat/lng are WGS84.
st = pd.DataFrame(
    [
        {
            "station": s["code"],
            "name": s["name"],
            "lat": float(s["lat"]),
            "lng": float(s["lng"]),
            "group": s["stationgroups"][0] if s["stationgroups"] else None,
            "height": s["measuringHeight"],
        }
        for s in stations
    ]
).set_index("station")

print(st["group"].value_counts())
print("heights published:", st["height"].notna().sum(), "of", len(st))
print(st[["lat", "lng"]].agg(["min", "max"]))
st

group
traffic       6
suburb        5
background    4
Name: count, dtype: int64
heights published: 3 of 15
          lat       lng
min  52.39840  13.22514
max  52.65327  13.64705


,name,lat,lng,group,height
station,,,,,
mc010,010 Wedding,52.542910,13.349260,background,NaN
mc027,027 Marienfelde,52.398400,13.368075,suburb,NaN
mc032,032 Grunewald,52.473190,13.225140,suburb,3.0
mc042,042 Neukölln,52.489451,13.430844,background,NaN
mc077,077 Buch,52.644100,13.483300,suburb,NaN
mc085,085 Friedrichshagen,52.447697,13.647050,suburb,NaN
mc117,117 Schildhornstraße,52.463620,13.318320,traffic,NaN
mc124,124 Mariendorfer Damm,52.438115,13.387720,traffic,NaN
mc145,145 Frohnau,52.653270,13.296080,suburb,NaN


In [8]:
# Pairwise great-circle distances in km. Haversine on WGS84 coordinates;
# over 40 km a spherical approximation is accurate to a few metres.
import numpy as np

lat = np.radians(st["lat"].to_numpy())
lng = np.radians(st["lng"].to_numpy())

dlat = lat[:, None] - lat[None, :]
dlng = lng[:, None] - lng[None, :]
a = np.sin(dlat / 2) ** 2 + np.cos(lat[:, None]) * np.cos(lat[None, :]) * np.sin(dlng / 2) ** 2
dist = pd.DataFrame(
    6371.0 * 2 * np.arcsin(np.sqrt(a)), index=st.index, columns=st.index
)

# Nearest other station per station: how dense is the network, really?
nearest = dist.replace(0, np.nan).min()
print(nearest.round(1).sort_values())
print("median nearest neighbour:", round(nearest.median(), 1), "km")

station
mc042     0.6
mc221     0.6
mc144     2.0
mc171     2.5
mc190     2.5
mc174     3.5
mc010     4.3
mc027     4.6
mc124     4.6
mc282     5.1
mc117     5.5
mc032     6.4
mc085     9.0
mc077    12.7
mc145    12.7
dtype: float64
median nearest neighbour: 4.6 km


Three stations publish a measuring height: 40 m, 35 m, 3 m. Twelve report null.
`stationgroups` matches the hard-coded TYPE dict in 02 and 03 (6 traffic,
5 suburb, 4 background), so the grouping is the operator's, not ours.